In [1]:
import pandas as pd


In [2]:
# Read in RASMI material intensity data and MI from old IMAGE-Materials CP
mi_rasmi = pd.read_excel("MI_ranges_20230905.xlsx", index_col=[0, 1, 2, 3, 4], 
                         sheet_name=["concrete", "brick", "wood", "steel", "glass", "plastics", "aluminum", "copper"])

mi_image_mat = pd.read_csv("Building_materials.csv", index_col=[0, 1, 2])


In [3]:
mi_image_mat.index.get_level_values('Region').unique()
mi_rasmi.get('concrete').index.get_level_values('function').unique()

Index(['NR', 'RM', 'RS'], dtype='object', name='function')

In [4]:
mi_image_mat

Steel  Concrete   Wood  Copper  Aluminium  Glass  \
Year Region Building_type                                                      
2020 1      1               32.47    861.52  50.91    1.73       6.42   2.68   
            2               32.89   1208.13  34.97    0.01       0.23   1.07   
            3               97.36    995.92  37.17    0.31       1.94   6.35   
            4               81.38    975.28  41.57    0.01       2.20   2.81   
     2      1               19.95    659.21  50.76    1.26       2.70   3.00   
...                           ...       ...    ...     ...        ...    ...   
2050 25     4              116.98    910.21  54.48    0.01       2.20   4.42   
     26     1               32.63    846.33  53.07    1.73       3.56   2.68   
            2               32.89   1208.13  34.97    0.01       0.23   1.07   
            3               97.36    995.92  37.17    0.31       1.94   6.35   
            4              116.98    910.21  54.48    0.01       2.20   4.42   

                            Brick  
Year Region Building_type          
2020 1      1              373.20  
            2              307.15  
            3              153.58  
            4               76.79  
     2      1              373.20  
...                           ...  
2050 25     4               76.79  
     26     1              373.20  
            2              307.15  
            3              153.58  
            4               76.79  

[208 rows x 7 columns]

In [5]:
# dictionary that maps RASMI R32 Regions to IMAGE R26 Regions

image_to_rasmi = {
    1: 'OECD_CAN',
    2: 'OECD_USA',
    3: 'LAM_MEX',
    4: 'LAM_LAM-L',
    5: 'LAM_BRA',
    6: 'LAM_LAM-L',
    7: ['MAF_MEA-H', 'MAF_MEA-M', 'MAF_NAF'],
    8: ['MAF_SSA-L', 'MAF_SSA-M'],
    9: ['MAF_SSA-L', 'MAF_SSA-M'],
    10: 'MAF_SAF',
    11: ['OECD_EFTA', 'OECD_EU12-H', 'OECD_EU12-M', 'OECD_EU15'],
    12: 'REF_EEU-FSU',
    13: 'OECD_TUR',
    14: ['REF_EEU-FSU', 'ASIA_TWN'],
    15: ['OECD_EEU', 'REF_CAS'],
    16: 'REF_RUS',
    17: ['MAF_MEA-H', 'MAF_MEA-M'],
    18: ['ASIA_IND', 'ASIA_OAS-CPA', 'ASIA_PAK'],
    19: 'OECD_KOR',
    20: 'ASIA_CHN',
    21: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    22: 'ASIA_IDN',
    23: 'OECD_JPN',
    24: 'OECD_AUNZ',
    25: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    26: ['MAF_SSA-L', 'MAF_SSA-M']
}

housing_type_image_to_rasmi = {
    1 : 'RS', #detached house to residential single family
    2 : 'RS', # semi detached house to residential single family
    3 : 'RM', # appartement to residential multi-family
    4 : 'RM', # high-rise to residential multi-family
}

housing_type_rasmi_to_image = {
    'RS' : [1, 2], # detached house and semi detached house to
    'RM' : [3, 4]  # appartement and high-rise to
}

housing_type_to_rasmi_building_structure = {
    1 : ['C', 'M', 'S', 'T'], # assumption that detached housing are average all structures
    2 : ['C', 'M', 'S', 'T'], # assumption that semi detached housing are average all structures
    3 : ['C', 'S'], # assumption that appartement are only made out of cement and steel structures
    4 : ['S'] # assumption that high-rise only steel structures
}

In [6]:
# material_list_rasmi = ["steel", "concrete", "wood", "copper", "aluminium", "glass", "brick"]
# mis_list_target = ["Steel", "Concrete", "Wood", "Copper", "Aluminium", "Glass", "Brick"]


# concrete_values = mi_rasmi.get("concrete").loc[:, "p_50"]

# # loop through all IMAGE classes and the according rasmi regions
# for image_class, rasmi_region in image_to_rasmi.items():
#     # convert str to list is necessary:
#     if isinstance(rasmi_region, str):
#         rasmi_region = [rasmi_region]
        
#     # loop through IMAGE regions and get the mean concrete mi value for each region for RS and RM (housing types)
#     for housingtype in set(housing_type_image_to_rasmi.values()): # RS and RM)

#         filtered = concrete_values[concrete_values.index.get_level_values('R5_32').isin(rasmi_region)
#                                 & concrete_values.index.get_level_values('function').isin([housingtype])]
#         print(filtered)

#         mean = filtered.mean()
#         mi_image_mat.loc[(2020, image_class, 1), "Concrete" ] = mean
#         print(image_class, mean)
#     break


In [7]:
material_list_rasmi = ["steel", "concrete", "wood", "copper", "aluminum", "glass", "brick"]
mis_list_target = ["Steel", "Concrete", "Wood", "Copper", "Aluminium", "Glass", "Brick"]

def replace_old_mis_with_rasmi(mi_rasmi: pd.DataFrame, material_list_rasmi: list, start_year: int = 2020, 
                               target_year: int = 2050, data_value = "p_50"):
    """    
    Replace old material intensities in IMAGE-Materials with RASMI values for concrete.
    
    material_name: str, non capitalised from rasmi
    mis_list_target_name: str, capitalised name of the material intensity in IMAGE-Materials
    data_value: str, can also be 0, 25, 75 or 100 percentile
    """
    for material_name in material_list_rasmi:
        # Capitalize material name
        material_name_capitalized = material_name.capitalize()
        # Replace if material_name is aluminum to Aluminium because of different spelling
        if material_name == "aluminum":
            material_name_capitalized = "Aluminium"

        print(material_name_capitalized, material_name)
        material_intensities = mi_rasmi.get(material_name).loc[:, data_value]

        # loop through all IMAGE classes and the according rasmi regions
        for image_region, rasmi_region in image_to_rasmi.items():
            # convert str to list is necessary:
            if isinstance(rasmi_region, str):
                rasmi_region = [rasmi_region]
            
            # loop through IMAGE regions and get the mean concrete mi value for each region for RS and RM (housing types)
            for housingtype_image, housingtype_rasmi in housing_type_image_to_rasmi.items():

                filtered_mis = material_intensities[material_intensities.index.get_level_values('R5_32').isin(rasmi_region) #filter for the right region
                                        & material_intensities.index.get_level_values('function').isin([housingtype_rasmi]) # filter for the right housing type of rasmi
                                        & material_intensities.index.get_level_values('structure').isin(housing_type_to_rasmi_building_structure[housingtype_image])] # filter for the right building structure

                mean_mi_value = filtered_mis.mean()
                # replace old values with new values
                mi_image_mat.loc[([start_year, target_year], image_region, housingtype_image), material_name_capitalized] = mean_mi_value
                # save as csv
    mi_image_mat.to_csv("Building_materials_rasmi.csv")
    print("done")





In [8]:
concrete = replace_old_mis_with_rasmi(mi_rasmi, material_list_rasmi)

Steel steel
Concrete concrete
Wood wood
Copper copper
Aluminium aluminum
Glass glass
Brick brick
done


In [9]:
mi_rasmi.get("concrete").loc[:, "p_50"]

     function  structure  R5_32         R5  
0    NR        C          ASIA_CHN      ASIA     945.360000
1    NR        C          ASIA_IDN      ASIA    1062.485000
2    NR        C          ASIA_IND      ASIA    1062.485000
3    NR        C          ASIA_OAS-CPA  ASIA    1062.485000
4    NR        C          ASIA_OAS-L    ASIA    1091.055000
                                                   ...     
379  RS        T          OECD_TUR      OECD     351.816073
380  RS        T          OECD_USA      OECD     277.275302
381  RS        T          REF_CAS       REF      351.177527
382  RS        T          REF_EEU-FSU   REF      351.177527
383  RS        T          REF_RUS       REF      351.177527
Name: p_50, Length: 384, dtype: float64

In [10]:
mi_image_mat

Steel    Concrete       Wood    Copper  \
Year Region Building_type                                               
2020 1      1              43.303105  574.250973  50.041437  0.182895   
            2              43.303105  574.250973  50.041437  0.182895   
            3              55.383413  762.544862  20.600000  0.182895   
            4              70.500000  534.983854  22.000000  0.182895   
     2      1              42.327828  481.434722  48.768730  0.182895   
...                              ...         ...        ...       ...   
2050 25     4              71.000000  734.855156  22.000000  0.182895   
     26     1              29.821679  572.352907  50.201717  0.182895   
            2              29.821679  572.352907  50.201717  0.182895   
            3              60.400573  821.157646  20.600000  0.182895   
            4              70.000000  677.167328  22.000000  0.182895   

                           Aluminium     Glass       Brick  
Year Region Building_type                                   
2020 1      1               0.489213  3.023166  244.713738  
            2               0.489213  3.023166  244.713738  
            3               0.491529  2.825000  139.333333  
            4               0.490000  2.650000   32.666667  
     2      1               0.490000  4.005891  240.375000  
...                              ...       ...         ...  
2050 25     4               0.490000  2.000000  107.000000  
     26     1               0.490765  3.206462  271.712209  
            2               0.490765  3.206462  271.712209  
            3               0.490000  2.234286  159.795751  
            4               0.490000  2.000000  107.000000  

[208 rows x 7 columns]